In [2]:
#import the necessary libraries
import cv2 as cv
import numpy as np
import mediapipe as mp
from insightface.app import FaceAnalysis
from ultralytics import YOLO
import time

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\HP\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [18]:
class RealTimeProctor:
    def __init__(self, face_db_path = None, yolo_model_path = 'yolov8n.pt', max_faces = 2):
        #this loads the insight face model using CPU
        self.face_app = FaceAnalysis(
            name="buffalo_1",
            root=r"C:\Users\HP\Desktop\notebooks\ican_hackathon\ICAN-TESTA-EXAMHACK",
            providers=["CPUExecutionProvider"]
)

        #this prepares the face model for inference witha given detection size
        self.face_app.prepare(ctx_id = 0, det_size = (320, 320))

        #this loads the YOLOv model for detecting suspicious objects like phones.
        self.yolo = YOLO(yolo_model_path)

        #this sets up mediapipe face mesh for detecting face landmarks to determine the head orientation
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(static_image_mode = False, max_num_faces = max_faces)

        #load webcam
        self.cap = cv.VideoCapture(0)
        self.face_db = self._load_face_db(face_db_path) if face_db_path else []

    def _load_face_db(self, path):
        #load known face embeddings from a saved .npy file or the database file
        return []
    
    #face recognition method: it compares the face embedding of the current detected face with those in self.face_db
    def _recognize_face(self, face):
        if not self.face_db:
            return "Unknown User"
        min_dist = float("inf")
        identity = "Unknown User"
        #loops throguh the known embeddings & calculates the euclidean distance between them and the current embedding
        #and if the distance is below 1.0 the current and the known embedding is considered a mach
        for entry in self.face_db:
            dist = np.linalg.norm(face.embedding - entry["embedding"])
            if dist < min_dist and dist < 1.0: #threshold for face match
                min_dist = dist #this saves the lowest distance 
                identity = entry["name"]
        return identity
    
    #head pose estimation: a simplified way to check if the person is looking away
    def _is_looking_away(self, landmarks):
        #use facial landmarks to determine whether the student is looking away
        left_eye = landmarks[33] #tip of the left eye
        right_eye = landmarks[263] #tip of the right eye
        dx = abs(left_eye.x - right_eye.x)
        return dx < 0.1  #if the eyes are too close together in 2D this might me the face has been turned

    #object detection: loops through the objects detected in frame and if the object is in the 
    # suspivuous list, it is flagged as suspicious
    def _detect_objects(self, frame):
        results = self.yolo(frame, verbose= False)[0]
        suspicious = []
        for box in results.boxes.data:
            cls = int(box[5])
            label = self.yolo.model.names[cls]
            if label in ["cell phone", "book", "person", "hand", "calculator"]:
                suspicious.append(label)
        return suspicious
    
    #main proctor
    def run(self):
        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break

            frame_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
            h, w = frame.shape[:2]

            #detect and recognize faces
            faces = self.face_app.get(frame)
            if len(faces) > 1:
                cv.putText(frame, "Warning: Multiple faces detected!", (20, 50), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            for face in faces:
                if face is None or face.box is None:
                    continue  # Skip bad detections

                box = face.box.astype(int)
                name = self._recognize_face(face)
                cv.rectangle(frame, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
                cv.putText(frame, name, (box[0], box[1]-10), cv.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

            
            #head pose detection
            mesh_results = self.face_mesh.process(frame_rgb)
            if mesh_results.multi_face_landmarks:
                for landmarks in mesh_results.multi_face_landmarks:
                    if self._is_looking_away(landmarks.landmark):
                        cv.putText(frame, "Warning: User looking away!", (20, 100), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 165, 255), 2)

            #object detection
            suspicious_items = self._detect_objects(frame)
            if suspicious_items:
                cv.putText(frame, f"Detected: {', '.join(set(suspicious_items))}", (20, 150), cv.FONT_HERSHEY_SIMPLEX, 1, (0,0, 255), 2)

            #Display
            cv.imshow("Proctoring Monitor", frame)
            if cv.waitKey(1) & 0xFF == ord('q'):
                break

        self.cap.release()
        cv.destroyAllWindows()

In [19]:
if __name__ == '__main__':
    face_db_path = "https://i.postimg.cc/Gt1F6PLH/my-pic.jpg"
    proctor = RealTimeProctor()
    proctor.run()

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\HP\Desktop\notebooks\ican_hackathon\ICAN-TESTA-EXAMHACK\models\buffalo_1\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\HP\Desktop\notebooks\ican_hackathon\ICAN-TESTA-EXAMHACK\models\buffalo_1\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\HP\Desktop\notebooks\ican_hackathon\ICAN-TESTA-EXAMHACK\models\buffalo_1\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\HP\Desktop\notebooks\ican_hackathon\ICAN-TESTA-EXAMHACK\models\buffalo_1\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with 